Step 1 — Understand the experiment
Query
  ↓
Embedding
  ↓
Compare against every stored vector
  ↓
Calculate similarity
  ↓
Sort by similarity
  ↓
Top-K

Step 2 — Create a small synthetic experiment

# Week 7 Day 2 — Qdrant Tour

## Topic 1: Linear Scan and Retrieval Scalability

A linear scan compares a query vector against every stored vector.
This is exact retrieval, but the amount of work grows with the number
of stored vectors.

In [1]:
import numpy as np
import time

print("NumPy version:", np.__version__)

NumPy version: 1.26.4


In [2]:
# Simulate a vector database containing 10,000 vectors

DIMENSIONS = 1536
NUM_VECTORS = 10_000

rng = np.random.default_rng(42)

vectors = rng.random((NUM_VECTORS, DIMENSIONS), dtype=np.float32)
query_vector = rng.random(DIMENSIONS, dtype=np.float32)

print("Vector matrix shape:", vectors.shape)
print("Query vector shape:", query_vector.shape)

Vector matrix shape: (10000, 1536)
Query vector shape: (1536,)


What we've just created

Think of this as a simulated vector database:
10,000 documents
       ↓
10,000 embedding vectors
       ↓
Each vector = 1,536 dimensions

And we have one query vector that we want to search against all 10,000 vectors.

We are deliberately using synthetic vectors here. The purpose is not semantic retrieval; it's to measure the computational work involved in exhaustive search.

Run this cell and send me the output. Then we'll perform the actual similarity calculation and measure how long the linear scan takes.

Now we'll perform the actual exhaustive/linear similarity search.
Step 3 — Linear scan

In [3]:
# Exact linear scan using dot product

start = time.perf_counter()

scores = vectors @ query_vector

elapsed_ms = (time.perf_counter() - start) * 1000

top_k = 5
top_indices = np.argsort(scores)[-top_k:][::-1]

print(f"Compared against {NUM_VECTORS:,} vectors")
print(f"Time taken: {elapsed_ms:.2f} ms")
print("Top-5 vector indices:", top_indices)
print("Top-5 scores:", scores[top_indices])

Compared against 10,000 vectors
Time taken: 3.45 ms
Top-5 vector indices: [5121 4828 6468 7312 6801]
Top-5 scores: [410.32373 404.95593 403.54602 403.4765  403.28915]


In [4]:
NUM_VECTORS = 100_000

vectors_100k = rng.random(
    (NUM_VECTORS, DIMENSIONS),
    dtype=np.float32
)

start = time.perf_counter()

scores_100k = vectors_100k @ query_vector

elapsed_ms = (time.perf_counter() - start) * 1000

print(f"Compared against {NUM_VECTORS:,} vectors")
print(f"Time taken: {elapsed_ms:.2f} ms")

Compared against 100,000 vectors
Time taken: 18.66 ms


Next: make the benchmark more reliable

A single timing can be noisy. Before moving to HNSW, let's measure each size several times and use the median.

In [5]:
def benchmark_linear_scan(vectors, query_vector, repeats=5):
    times_ms = []

    for _ in range(repeats):
        start = time.perf_counter()
        _ = vectors @ query_vector
        elapsed_ms = (time.perf_counter() - start) * 1000
        times_ms.append(elapsed_ms)

    return times_ms


times_10k = benchmark_linear_scan(vectors, query_vector)
times_100k = benchmark_linear_scan(vectors_100k, query_vector)

print("10,000 vectors:")
print("Times (ms):", [round(t, 2) for t in times_10k])
print("Median (ms):", round(float(np.median(times_10k)), 2))

print()

print("100,000 vectors:")
print("Times (ms):", [round(t, 2) for t in times_100k])
print("Median (ms):", round(float(np.median(times_100k)), 2))

10,000 vectors:
Times (ms): [1.22, 1.02, 0.92, 0.88, 0.85]
Median (ms): 0.92

100,000 vectors:
Times (ms): [10.79, 9.98, 9.99, 10.1, 9.95]
Median (ms): 9.99


Day 2 — Topic 2: HNSW
Concept first

HNSW = Hierarchical Navigable Small World graph.

Query
  │
  ├── compare with Vector 1
  ├── compare with Vector 2
  ├── compare with Vector 3
  ├── ...
  └── compare with Vector 100,000

            Layer 2
        A ─────── D
        │         │
        │         │
          Layer 1
      A ─ B ─ C ─ D ─ E
          │     │
          F ─── G

Your next small lab step

Let's inspect the HNSW configuration of the Qdrant collection we created on Day 1.

In [7]:
import os
from getpass import getpass

os.environ["QDRANT_URL"] = input("Enter QDRANT_URL: ").strip()
os.environ["QDRANT_API_KEY"] = getpass("Enter QDRANT_API_KEY: ")

print("QDRANT_URL configured:", bool(os.environ["QDRANT_URL"]))
print("QDRANT_API_KEY configured:", bool(os.environ["QDRANT_API_KEY"]))

QDRANT_URL configured: True
QDRANT_API_KEY configured: True


In [8]:
from qdrant_client import QdrantClient
import os

qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)

collection_name = "w7_animals_gorthi"

info = qdrant.get_collection(collection_name)

print("Collection:", collection_name)
print("Points:", info.points_count)
print("Vector size:", info.config.params.vectors.size)
print("Distance:", info.config.params.vectors.distance)
print("HNSW config:")
print(info.config.hnsw_config)

Collection: w7_animals_gorthi
Points: 10
Vector size: 1536
Distance: Cosine
HNSW config:
m=16 ef_construct=100 full_scan_threshold=10000 max_indexing_threads=0 on_disk=False memory=None payload_m=None inline_storage=None


Let's interpret your output
| Setting                | Your value | Meaning                                                         |
| ---------------------- | ---------: | --------------------------------------------------------------- |
| `m`                    |     **16** | Maximum number of graph connections considered per node         |
| `ef_construct`         |    **100** | Controls how much work is used when constructing the HNSW graph |
| `full_scan_threshold`  | **10,000** | Qdrant may use full/exhaustive scan below this threshold        |
| `max_indexing_threads` |      **0** | Qdrant determines the indexing thread usage                     |
| `on_disk`              |  **False** | HNSW index is kept in memory                                    |


Next small step: see HNSW in action

Let's inspect whether Qdrant is actually using the collection's index and compare exact/full-scan vs indexed retrieval.

In [9]:
# Inspect the collection before searching

info = qdrant.get_collection(collection_name)

print("Collection:", collection_name)
print("Points:", info.points_count)
print("Indexed vectors:", info.indexed_vectors_count)
print("HNSW config:", info.config.hnsw_config)

Collection: w7_animals_gorthi
Points: 10
Indexed vectors: 0
HNSW config: m=16 ef_construct=100 full_scan_threshold=10000 max_indexing_threads=0 on_disk=False memory=None payload_m=None inline_storage=None


Next Day 2 topic: Quantization
Now let's move to quantization, which is another technique covered in the Day 2 lab. The guide introduces quantization as a way to reduce vector memory usage and discusses the trade-off between resource savings and retrieval quality.

Before touching Qdrant configuration, let's understand it with a tiny Python experiment.

Run this one cell

In [11]:
import numpy as np

vectors_f32 = np.random.default_rng(42).random(
    (1000, 1536),
    dtype=np.float32
)

vectors_f16 = vectors_f32.astype(np.float16)

print("Float32:")
print("dtype:", vectors_f32.dtype)
print("bytes:", vectors_f32.nbytes)

print()

print("Float16:")
print("dtype:", vectors_f16.dtype)
print("bytes:", vectors_f16.nbytes)

print()

print("Memory reduction:",
      round((1 - vectors_f16.nbytes / vectors_f32.nbytes) * 100, 2),
      "%")

Float32:
dtype: float32
bytes: 6144000

Float16:
dtype: float16
bytes: 3072000

Memory reduction: 50.0 %


Next step: see the effect on similarity

In [1]:
# Compare cosine similarity before and after float16 conversion

a = vectors_f32[0]
b = vectors_f32[1]

def cosine_similarity(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

similarity_f32 = cosine_similarity(a, b)

a_f16 = a.astype(np.float16)
b_f16 = b.astype(np.float16)

similarity_f16 = cosine_similarity(a_f16, b_f16)

print("Cosine similarity using float32:", similarity_f32)
print("Cosine similarity using float16:", similarity_f16)
print("Absolute difference:", abs(similarity_f32 - similarity_f16))

NameError: name 'vectors_f32' is not defined

In [2]:
import numpy as np

# Recreate the same vectors
vectors_f32 = np.random.default_rng(42).random(
    (1000, 1536),
    dtype=np.float32
)

# Select two vectors
a = vectors_f32[0]
b = vectors_f32[1]

# Cosine similarity function
def cosine_similarity(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

# Similarity using float32
similarity_f32 = cosine_similarity(a, b)

# Convert to float16
a_f16 = a.astype(np.float16)
b_f16 = b.astype(np.float16)

# Similarity using float16
similarity_f16 = cosine_similarity(a_f16, b_f16)

print("Cosine similarity using float32:", similarity_f32)
print("Cosine similarity using float16:", similarity_f16)
print("Absolute difference:", abs(similarity_f32 - similarity_f16))

Cosine similarity using float32: 0.7434979
Cosine similarity using float16: 0.7437
Absolute difference: 0.00015443563


Day 2 — Next Topic: Similarity Metrics

These are different ways of measuring how close two vectors are. The Week 7 guide specifically calls out these three metrics and includes a "silent bug" demonstration involving normalization and dot product.

The same vectors can produce different rankings depending on the similarity metric and whether the vectors are normalized.

In [3]:
import numpy as np

# Three simple vectors
query = np.array([1.0, 1.0])

vector_a = np.array([2.0, 2.0])
vector_b = np.array([1.0, 0.0])
vector_c = np.array([0.0, 1.0])

def cosine_similarity(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

def euclidean_distance(x, y):
    return np.linalg.norm(x - y)

vectors = {
    "A": vector_a,
    "B": vector_b,
    "C": vector_c,
}

for name, vector in vectors.items():
    print(f"{name}:")
    print(f"  Dot product:       {np.dot(query, vector):.4f}")
    print(f"  Cosine similarity: {cosine_similarity(query, vector):.4f}")
    print(f"  L2 distance:       {euclidean_distance(query, vector):.4f}")

A:
  Dot product:       4.0000
  Cosine similarity: 1.0000
  L2 distance:       1.4142
B:
  Dot product:       1.0000
  Cosine similarity: 0.7071
  L2 distance:       1.0000
C:
  Dot product:       1.0000
  Cosine similarity: 0.7071
  L2 distance:       1.0000


Step 1 — Create the controlled example

In [4]:
import numpy as np

# Query vector
query = np.array([1.0, 1.0])

# Vector A: same direction as query, smaller magnitude
vector_a = np.array([2.0, 2.0])

# Vector B: somewhat different direction, but much larger magnitude
vector_b = np.array([10.0, 1.0])

def cosine_similarity(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

print("Query:", query)
print("Vector A:", vector_a)
print("Vector B:", vector_b)

print("\nDot product:")
print("A:", np.dot(query, vector_a))
print("B:", np.dot(query, vector_b))

print("\nCosine similarity:")
print("A:", round(cosine_similarity(query, vector_a), 4))
print("B:", round(cosine_similarity(query, vector_b), 4))

Query: [1. 1.]
Vector A: [2. 2.]
Vector B: [10.  1.]

Dot product:
A: 4.0
B: 11.0

Cosine similarity:
A: 1.0
B: 0.774


Step 2 — Fix the problem through normalization

In [5]:
def normalize(vector):
    return vector / np.linalg.norm(vector)

query_norm = normalize(query)
a_norm = normalize(vector_a)
b_norm = normalize(vector_b)

print("Normalized vectors:")
print("Query:", query_norm)
print("A:", a_norm)
print("B:", b_norm)

print("\nDot product AFTER normalization:")
print("A:", round(np.dot(query_norm, a_norm), 4))
print("B:", round(np.dot(query_norm, b_norm), 4))

print("\nCosine similarity:")
print("A:", round(cosine_similarity(query, vector_a), 4))
print("B:", round(cosine_similarity(query, vector_b), 4))

Normalized vectors:
Query: [0.70710678 0.70710678]
A: [0.70710678 0.70710678]
B: [0.99503719 0.09950372]

Dot product AFTER normalization:
A: 1.0
B: 0.774

Cosine similarity:
A: 1.0
B: 0.774


Day 2 — Final Topic: Mini-RAG

User query
    ↓
Create query embedding
    ↓
Vector similarity search
    ↓
Retrieve relevant chunks
    ↓
Use retrieved context
    ↓
Generate an answer

Step 1 — Let's use your existing Qdrant animal collection

In [6]:
from openai import OpenAI

client = OpenAI()

query = "Which animals are good examples of pets?"

query_vector = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding

print("Query:", query)
print("Embedding dimensions:", len(query_vector))

Query: Which animals are good examples of pets?
Embedding dimensions: 1536


Step 2 — Retrieve the relevant chunks from Qdrant

In [7]:
results = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=3,
    with_payload=True,
).points

print("Top-3 retrieved results:\n")

for i, result in enumerate(results, start=1):
    print(f"{i}. {result.payload['animal_id']}")
    print(f"   Category: {result.payload['category']}")
    print(f"   Score: {result.score:.4f}")
    print(f"   Text: {result.payload['text']}")
    print()

NameError: name 'qdrant' is not defined

Step 2 — Reconnect to Qdrant and retrieve

In [9]:
import os
from getpass import getpass

os.environ["QDRANT_URL"] = input("Enter QDRANT_URL: ").strip()
os.environ["QDRANT_API_KEY"] = getpass("Enter QDRANT_API_KEY: ")

print("QDRANT_URL configured:", bool(os.environ["QDRANT_URL"]))
print("QDRANT_API_KEY configured:", bool(os.environ["QDRANT_API_KEY"]))

QDRANT_URL configured: True
QDRANT_API_KEY configured: True


In [10]:
import os
from qdrant_client import QdrantClient
from openai import OpenAI

# Connect to Qdrant
qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)

# OpenAI client
client = OpenAI()

# Collection
collection_name = "w7_animals_gorthi"

# Query
query = "Which animals are good examples of pets?"

# Create query embedding
query_vector = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding

# Retrieve top-3
results = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=3,
    with_payload=True,
).points

print("Query:", query)
print("Embedding dimensions:", len(query_vector))
print("\nTop-3 retrieved results:\n")

for i, result in enumerate(results, start=1):
    print(f"{i}. {result.payload['animal_id']}")
    print(f"   Category: {result.payload['category']}")
    print(f"   Score: {result.score:.4f}")
    print(f"   Text: {result.payload['text']}")
    print()

Query: Which animals are good examples of pets?
Embedding dimensions: 1536

Top-3 retrieved results:

1. dog
   Category: mammal
   Score: 0.5019
   Text: Dogs are domesticated mammals commonly kept as pets and are known for their loyalty and companionship.

2. cat
   Category: mammal
   Score: 0.4696
   Text: Cats are domesticated mammals often kept as pets and are known for their independence and hunting instincts.

3. gecko
   Category: reptile
   Score: 0.3979
   Text: Geckos are small reptiles that are often kept as pets and are known for their ability to climb surfaces.



Final Mini-RAG step — Generate a grounded answer

In [11]:
context = "\n".join(
    f"- {result.payload['animal_id']}: {result.payload['text']}"
    for result in results
)

prompt = f"""Answer the question using only the retrieved context below.

Question:
{query}

Retrieved context:
{context}

If the answer is not supported by the context, say so.
"""

response = client.responses.create(
    model="gpt-4o-mini",
    input=prompt,
)

print("Question:")
print(query)

print("\nRetrieved context:")
print(context)

print("\nGenerated answer:")
print(response.output_text)

Question:
Which animals are good examples of pets?

Retrieved context:
- dog: Dogs are domesticated mammals commonly kept as pets and are known for their loyalty and companionship.
- cat: Cats are domesticated mammals often kept as pets and are known for their independence and hunting instincts.
- gecko: Geckos are small reptiles that are often kept as pets and are known for their ability to climb surfaces.

Generated answer:
Good examples of pets are dogs, cats, and geckos.
